# NLRB Data Integration

## What is the NLRB?

The **National Labor Relations Board** (NLRB) is an independent federal agency created in 1935
by the National Labor Relations Act. It protects the rights of private-sector employees to
organize, engage in collective bargaining, and take collective action such as strikes.

The NLRB divides the United States into **26 active regions** (numbered 1--31 with gaps due to
historical consolidations). Each region has a regional office that processes unfair labor
practice charges and representation petitions for its territory.

### Why geographic coverage matters for labor organizing

Understanding NLRB region boundaries is essential for:

- **Filing petitions**: Workers must file with the correct regional office.
- **Jurisdictional analysis**: Some states are split across multiple regions (e.g., New York
  spans Regions 2, 3, and 29), while others share a region with neighboring states.
- **Resource allocation**: Regional offices vary in caseload; geographic analysis helps
  identify overburdened regions.
- **Cross-referencing with Census data**: NLRB regions can be joined to Census state
  boundaries via FIPS codes to overlay demographic, economic, and workforce data.

The `siege_utilities` library includes a built-in NLRB module with:

- A **static registry** (`NLRB_REGIONS` dict) mapping region numbers to office cities and states
- A **Django model** (`NLRBRegion`) for persistent storage with optional dissolved geometry
- A **population service** (`NLRBPopulationService`) and management command for database loading

This notebook explores the NLRB data **without** requiring Django or a database.

In [ ]:
import sys
from pathlib import Path
from collections import Counter, defaultdict

# Ensure siege_utilities is importable
sys.path.insert(0, str(Path.cwd().parent))

import siege_utilities as su
su.configure_shared_logging(level="INFO")

su.log_info("NLRB Data Integration notebook initialized.")

## 2. NLRB Regions Overview

The `NLRB_REGIONS` dictionary is the authoritative registry of active NLRB regions.
Each entry maps a region number to its office city and the list of state/territory
abbreviations it covers.

Note: some regions share states at the county level (e.g., Regions 2, 3, and 29 all
list NY because they each cover different parts of the state). The NLRB does not
publish county-level shapefiles, so state-level coverage is the best publicly available
approximation.

In [ ]:
from siege_utilities.geo.django.services.nlrb_service import NLRB_REGIONS

# Display as a formatted table
header = f"{'Region':>8}  {'Office':<20}  {'States Covered'}"
print(header)
print("-" * len(header))

for region_num in sorted(NLRB_REGIONS.keys()):
    info = NLRB_REGIONS[region_num]
    states_str = ", ".join(sorted(info["states"]))
    print(f"{region_num:>8}  {info['office']:<20}  {states_str}")

print(f"\nTotal regions: {len(NLRB_REGIONS)}")

## 3. Region Data Exploration

Let us examine the coverage statistics: how many regions exist, how many states each
region covers, and which states appear in multiple regions.

In [ ]:
# --- Coverage statistics ---

# States per region
print("States per region:")
print(f"{'Region':>8}  {'Count':>5}  States")
print("-" * 60)
for region_num in sorted(NLRB_REGIONS.keys()):
    states = NLRB_REGIONS[region_num]["states"]
    print(f"{region_num:>8}  {len(states):>5}  {', '.join(sorted(states))}")

# Summary statistics
counts = [len(info["states"]) for info in NLRB_REGIONS.values()]
print(f"\nMin states per region: {min(counts)}")
print(f"Max states per region: {max(counts)}")
print(f"Mean states per region: {sum(counts) / len(counts):.1f}")

In [ ]:
# Which states appear in multiple regions?
state_to_regions: dict[str, list[int]] = defaultdict(list)
for region_num, info in NLRB_REGIONS.items():
    for state in info["states"]:
        state_to_regions[state].append(region_num)

# All unique states/territories covered
all_states = sorted(state_to_regions.keys())
print(f"Total unique states/territories covered: {len(all_states)}")
print(f"States: {', '.join(all_states)}\n")

# Multi-region states
print("States appearing in multiple regions:")
print(f"{'State':<6}  {'Regions'}")
print("-" * 40)
for state in sorted(state_to_regions.keys()):
    regions = state_to_regions[state]
    if len(regions) > 1:
        print(f"{state:<6}  {', '.join(str(r) for r in sorted(regions))}")

In [ ]:
# Office city frequency — which cities host multiple regional offices?
office_counter = Counter(info["office"] for info in NLRB_REGIONS.values())
print("Office cities hosting multiple regions:")
for city, count in office_counter.most_common():
    if count > 1:
        regions = [r for r, info in NLRB_REGIONS.items() if info["office"] == city]
        print(f"  {city}: Regions {', '.join(str(r) for r in sorted(regions))}")

## 4. Django Model

The `NLRBRegion` model extends `TemporalBoundary`, which provides:

- `feature_id` (CharField) — stable identifier, auto-set to `NLRB-{number:02d}`
- `boundary_id` (CharField) — same as feature_id
- `name` (CharField) — human-readable label (e.g., "NLRB Region 5")
- `vintage_year` (PositiveSmallIntegerField) — edition year of the data
- `valid_from` / `valid_to` (DateField) — temporal validity range
- `source` (CharField) — auto-set to "NLRB"
- `geometry` (MultiPolygonField) — optional; built by dissolving county boundaries

NLRB-specific fields:

- `region_number` (PositiveSmallIntegerField, indexed) — e.g., 5
- `region_office` (CharField) — e.g., "Baltimore"
- `states_covered` (JSONField) — e.g., `["DC", "MD", "VA", "WV"]`

Constraints:

- `unique_together`: (`feature_id`, `vintage_year`)
- Indexes on `region_number` and `vintage_year`

Source: `siege_utilities/geo/django/models/federal.py`

In [ ]:
# Demonstrate the model structure without Django.
# We can inspect the source and simulate what an instance would look like.

import inspect

try:
    from siege_utilities.geo.django.models.federal import NLRBRegion

    # Show model fields
    print("NLRBRegion model fields:")
    print("-" * 60)
    for field in NLRBRegion._meta.get_fields():
        field_type = type(field).__name__
        print(f"  {field.name:<25} {field_type}")

    print(f"\nMeta unique_together: {NLRBRegion._meta.unique_together}")
    print(f"Meta indexes: {[str(idx) for idx in NLRBRegion._meta.indexes]}")
except Exception as e:
    su.log_info(f"Django not available ({e}); showing model structure from source.")

    # Fallback: display the model structure from the registry dict
    print("NLRBRegion model fields (from source inspection):")
    print("-" * 60)
    fields = [
        ("feature_id",     "CharField(max_length=60)",      "Auto: NLRB-{num:02d}"),
        ("boundary_id",    "CharField",                      "Same as feature_id"),
        ("name",           "CharField(max_length=255)",      "e.g. 'NLRB Region 5'"),
        ("vintage_year",   "PositiveSmallIntegerField",      "e.g. 2024"),
        ("valid_from",     "DateField(null=True)",           "Start of validity"),
        ("valid_to",       "DateField(null=True)",           "End of validity"),
        ("source",         "CharField",                      "Auto: 'NLRB'"),
        ("geometry",       "MultiPolygonField(null=True)",   "Optional dissolved geometry"),
        ("region_number",  "PositiveSmallIntegerField",      "e.g. 5"),
        ("region_office",  "CharField(max_length=100)",      "e.g. 'Baltimore'"),
        ("states_covered", "JSONField",                      "e.g. ['DC','MD','VA','WV']"),
    ]
    for name, ftype, desc in fields:
        print(f"  {name:<20} {ftype:<35} {desc}")

    print("\nConstraints:")
    print("  unique_together: (feature_id, vintage_year)")
    print("  indexes: [region_number], [vintage_year]")

In [ ]:
# Simulate creating an NLRBRegion instance (without database)
# This mirrors exactly what NLRBPopulationService.populate() builds.

region_num = 5
info = NLRB_REGIONS[region_num]

simulated_instance = {
    "feature_id": f"NLRB-{region_num:02d}",
    "boundary_id": f"NLRB-{region_num:02d}",
    "region_number": region_num,
    "region_office": info["office"],
    "states_covered": info["states"],
    "vintage_year": 2024,
    "source": "NLRB",
    "name": f"NLRB Region {region_num}",
    "geometry": None,  # Would be populated by dissolve_counties
}

print("Simulated NLRBRegion instance:")
for key, value in simulated_instance.items():
    print(f"  {key:<20} = {value!r}")

print(f"\n__str__ would return: 'NLRB Region {region_num} ({info['office']})'")

## 5. Population Service

The `NLRBPopulationService` handles creating/updating `NLRBRegion` records in the
database. It wraps the operation in a transaction and supports three modes:

1. **Default**: Create new records, skip existing ones.
2. **Update mode** (`update_existing=True`): Overwrite existing records with current data.
3. **Dissolve mode** (`dissolve_counties=True`): Build region geometry by dissolving
   county boundaries from the Census County model.

### Dissolve Workflow

When `dissolve_counties=True`, the service:

1. Looks up state FIPS codes for each state abbreviation in the region
   (using `STATE_FIPS_CODES` from `siege_utilities.config.census_constants`)
2. Queries `County` model for all counties whose GEOID starts with those FIPS codes
3. Iteratively unions all county geometries into a single dissolved shape
4. Wraps the result in a `MultiPolygon` if needed
5. Stores the dissolved geometry on the `NLRBRegion` record

This dissolve approach is necessary because the NLRB does not publish GIS shapefiles.

### Return Value

The service returns an `NLRBPopulationResult` dataclass with:
- `records_created` (int)
- `records_updated` (int)
- `records_skipped` (int)
- `errors` (list)
- `success` (property: `True` if no errors)

Source: `siege_utilities/geo/django/services/nlrb_service.py`

In [ ]:
# Show the service API (works without Django for inspection)
from siege_utilities.geo.django.services.nlrb_service import (
    NLRBPopulationService,
    NLRBPopulationResult,
)

# Inspect the populate method signature
sig = inspect.signature(NLRBPopulationService.populate)
print(f"NLRBPopulationService.populate{sig}")
print()

# Show the result dataclass
print("NLRBPopulationResult fields:")
result = NLRBPopulationResult()
for f in ["records_created", "records_updated", "records_skipped", "errors", "success"]:
    print(f"  {f:<20} = {getattr(result, f)!r}")

print("\nExample usage (requires Django):")
print("  service = NLRBPopulationService()")
print("  result = service.populate(vintage_year=2024)")
print("  result = service.populate(vintage_year=2024, update_existing=True)")
print("  result = service.populate(vintage_year=2024, dissolve_counties=True)")

## 6. Geographic Visualization

Visualize which states belong to which NLRB region using a color-coded US map.

Since some states are split across multiple regions at the county level, the map uses
the **first (lowest-numbered) region** for each state. States in multiple regions are
annotated in the legend.

In [ ]:
try:
    import matplotlib
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False
    su.log_info("matplotlib not available; skipping visualization.")

if HAS_MATPLOTLIB:
    try:
        import geopandas as gpd
        HAS_GEOPANDAS = True
    except ImportError:
        HAS_GEOPANDAS = False
        su.log_info("geopandas not available; using tabular fallback.")

In [ ]:
# Build a state -> primary region mapping (lowest region number wins for display)
state_primary_region: dict[str, int] = {}
for state in sorted(state_to_regions.keys()):
    state_primary_region[state] = min(state_to_regions[state])

if HAS_MATPLOTLIB and HAS_GEOPANDAS:
    # Use Census TIGER state boundaries from naturalearth or local shapefile
    try:
        from siege_utilities.config.census_constants import STATE_FIPS_CODES

        # Build reverse lookup: abbreviation -> FIPS
        abbr_to_fips = STATE_FIPS_CODES  # already {abbr: fips}

        # Load US states from the bundled naturalearth dataset
        world = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))  # type: ignore[attr-defined]
        # naturalearth doesn't have US states, so try the higher-res cultural dataset
        # Fallback: use a simplified approach with state centroids
        raise ImportError("Use simplified approach")
    except Exception:
        pass

    # Simplified visualization: plot state abbreviations colored by region
    # Use a grid layout approximating the US map
    # Standard US state grid positions (row, col) for cartographic approximation
    state_grid = {
        "AK": (0, 0), "ME": (0, 10),
        "WI": (1, 5), "VT": (1, 9), "NH": (1, 10),
        "WA": (2, 0), "MT": (2, 2), "ND": (2, 4), "MN": (2, 5),
        "MI": (2, 7), "NY": (2, 9), "MA": (2, 10),
        "OR": (3, 0), "ID": (3, 1), "WY": (3, 2), "SD": (3, 4),
        "IA": (3, 5), "IL": (3, 6), "IN": (3, 7), "OH": (3, 8),
        "PA": (3, 9), "CT": (3, 10), "RI": (3, 11),
        "CA": (4, 0), "NV": (4, 1), "UT": (4, 2), "CO": (4, 3),
        "NE": (4, 4), "MO": (4, 5), "KY": (4, 7), "WV": (4, 8),
        "VA": (4, 9), "NJ": (4, 10), "DE": (4, 11),
        "AZ": (5, 1), "NM": (5, 2), "KS": (5, 3), "AR": (5, 5),
        "TN": (5, 6), "NC": (5, 8), "MD": (5, 9), "DC": (5, 10),
        "OK": (6, 3), "LA": (6, 5), "MS": (6, 6), "AL": (6, 7),
        "GA": (6, 8), "SC": (6, 9),
        "TX": (7, 3), "FL": (7, 8),
        "HI": (8, 0), "PR": (8, 9), "VI": (8, 10),
    }

    # Assign colors: one per unique region
    unique_regions = sorted(set(state_primary_region.values()))
    cmap = plt.cm.get_cmap("tab20", len(unique_regions))
    region_colors = {r: cmap(i) for i, r in enumerate(unique_regions)}

    fig, ax = plt.subplots(figsize=(16, 10))
    ax.set_xlim(-1, 13)
    ax.set_ylim(-0.5, 9.5)
    ax.invert_yaxis()
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title("NLRB Regions by State (Primary Region Assignment)", fontsize=16, fontweight="bold")

    for state, (row, col) in state_grid.items():
        region = state_primary_region.get(state)
        if region is None:
            color = "lightgray"
        else:
            color = region_colors[region]

        rect = plt.Rectangle((col - 0.4, row - 0.4), 0.8, 0.8,
                             facecolor=color, edgecolor="white", linewidth=1.5)
        ax.add_patch(rect)
        ax.text(col, row, f"{state}\n{region or '?'}", ha="center", va="center",
                fontsize=7, fontweight="bold", color="black")

    # Legend (show a subset of regions to avoid clutter)
    legend_patches = [
        mpatches.Patch(color=region_colors[r], label=f"Region {r} ({NLRB_REGIONS[r]['office']})")
        for r in unique_regions
    ]
    ax.legend(handles=legend_patches, loc="lower left", fontsize=6, ncol=3,
              title="NLRB Region (Office City)", title_fontsize=8)

    plt.tight_layout()
    plt.show()

elif HAS_MATPLOTLIB:
    su.log_info("Displaying text-based region map (geopandas not available).")
    for state in sorted(state_primary_region.keys()):
        region = state_primary_region[state]
        multi = " *" if len(state_to_regions[state]) > 1 else ""
        print(f"  {state} -> Region {region} ({NLRB_REGIONS[region]['office']}){multi}")
    print("\n  * = state spans multiple regions")
else:
    su.log_info("No visualization libraries available. See text tables above.")

## 7. Integration with Census

NLRB regions can be linked to Census geographic boundaries through **state FIPS codes**.
The `siege_utilities.config.census_constants` module provides `STATE_FIPS_CODES`
(abbreviation to FIPS) and `FIPS_TO_STATE` (FIPS to abbreviation) dictionaries that
serve as the join key.

This allows overlaying NLRB regions onto Census state boundaries, aggregating
demographic data by NLRB region, and building dissolved region geometries from
county-level shapefiles.

In [ ]:
from siege_utilities.config.census_constants import STATE_FIPS_CODES, FIPS_TO_STATE

# Build an NLRB region -> FIPS codes mapping
print("NLRB Region -> State FIPS codes:")
print(f"{'Region':>8}  {'Office':<20}  {'FIPS Codes'}")
print("-" * 65)

for region_num in sorted(NLRB_REGIONS.keys()):
    info = NLRB_REGIONS[region_num]
    fips_codes = []
    for abbr in sorted(info["states"]):
        fips = STATE_FIPS_CODES.get(abbr)
        if fips:
            fips_codes.append(f"{abbr}={fips}")
        else:
            fips_codes.append(f"{abbr}=??")
    print(f"{region_num:>8}  {info['office']:<20}  {', '.join(fips_codes)}")

In [ ]:
# Demonstrate the join: given a state FIPS code, find which NLRB regions cover it

def fips_to_nlrb_regions(fips_code: str) -> list[dict]:
    """Given a state FIPS code, return the NLRB regions covering that state."""
    abbr = FIPS_TO_STATE.get(fips_code)
    if not abbr:
        return []
    return [
        {"region": r, "office": info["office"]}
        for r, info in NLRB_REGIONS.items()
        if abbr in info["states"]
    ]

# Examples
test_fips = ["36", "48", "06", "17"]
for fips in test_fips:
    state = FIPS_TO_STATE.get(fips, "??")
    regions = fips_to_nlrb_regions(fips)
    region_strs = [f"Region {r['region']} ({r['office']})" for r in regions]
    print(f"  FIPS {fips} ({state}): {', '.join(region_strs)}")

In [ ]:
# Show how this would work in a Django/Census context:
# Aggregate Census population data by NLRB region.

example_census_integration = """
# With Census demographic data loaded in Django:

from siege_utilities.geo.django.models import State, NLRBRegion
from siege_utilities.geo.django.models import DemographicSnapshot
from siege_utilities.config.census_constants import STATE_FIPS_CODES

# For each NLRB region, sum the population of its member states:
for region in NLRBRegion.objects.filter(vintage_year=2024):
    total_pop = 0
    for abbr in region.states_covered:
        fips = STATE_FIPS_CODES.get(abbr)
        if fips:
            state = State.objects.get(geoid=fips, vintage_year=2020)
            snapshot = state.demographics.filter(year=2022).first()
            if snapshot:
                total_pop += snapshot.total_population
    print(f"Region {region.region_number} ({region.region_office}): pop={total_pop:,}")
"""
print(example_census_integration)

## 8. Management Command

The `populate_nlrb_regions` management command provides a CLI interface to the
`NLRBPopulationService`.

### Usage

```bash
# Basic: create all 26 NLRB region records for vintage year 2024
python manage.py populate_nlrb_regions

# Specify a vintage year
python manage.py populate_nlrb_regions --year 2024

# Update existing records (overwrite office/states data)
python manage.py populate_nlrb_regions --year 2024 --update

# Build region geometry by dissolving county boundaries
# (requires populated County model in the database)
python manage.py populate_nlrb_regions --dissolve-counties

# Combine: update existing + dissolve geometry
python manage.py populate_nlrb_regions --year 2024 --update --dissolve-counties
```

### Arguments

| Argument | Default | Description |
|----------|---------|-------------|
| `--year` | 2024 | Vintage year for the region records |
| `--update` | False | Update existing records instead of skipping |
| `--dissolve-counties` | False | Build region geometry from county boundaries |

### Output

```
NLRB regions: created=26, updated=0, skipped=0
Done.
```

Source: `siege_utilities/geo/django/management/commands/populate_nlrb_regions.py`

In [ ]:
# Verify the command module is importable
try:
    from siege_utilities.geo.django.management.commands.populate_nlrb_regions import Command
    print(f"Command class: {Command.__name__}")
    print(f"Help text: {Command.help}")

    # Show arguments by inspecting add_arguments
    import argparse
    parser = argparse.ArgumentParser()
    cmd = Command()
    cmd.add_arguments(parser)
    print("\nArguments:")
    for action in parser._actions:
        if action.option_strings:
            print(f"  {', '.join(action.option_strings):<25} default={action.default!r:<10} {action.help}")
except Exception as e:
    su.log_info(f"Cannot import management command directly ({e}).")
    print("The command is available via: python manage.py populate_nlrb_regions")

## Summary

This notebook demonstrated the `siege_utilities` NLRB data integration module:

1. **NLRB_REGIONS registry**: 26 active regions with office cities and state coverage
2. **Coverage analysis**: Multi-region states, office distribution, coverage gaps
3. **NLRBRegion model**: Temporal boundary model with region-specific fields
4. **NLRBPopulationService**: Database population with optional geometry dissolve
5. **Census integration**: State FIPS codes as the join key between NLRB and Census data
6. **Management command**: CLI tool for one-command database population

### Related Notebooks

- **NB04** — Spatial Data & Census Boundaries
- **NB13** — GeoDjango Integration (includes NLRB section)
- **NB15** — Census Demographics Integration
- **NB20** — Multi-Source Spatial Tabulation (Census × NCES × NLRB × RDH)

### Key Files

| File | Purpose |
|------|----------|
| `siege_utilities/geo/django/services/nlrb_service.py` | NLRB_REGIONS dict, NLRBPopulationService |
| `siege_utilities/geo/django/models/federal.py` | NLRBRegion Django model |
| `siege_utilities/geo/django/management/commands/populate_nlrb_regions.py` | CLI command |
| `siege_utilities/config/census_constants.py` | STATE_FIPS_CODES, FIPS_TO_STATE |